# Turbopuffer RAG Training Pipeline

This notebook provides a streamlined 4-step process to generate synthetic QA data and launch a training job using Turbopuffer as the corpus backend:

1. **Point to dataset** - Chunk and upload documents to a Turbopuffer namespace
2. **Create synthetic QA** - Generate question-answer pairs
3. **Create env** - Configure the Turbopuffer search environment
4. **Run training job** - Launch the training

## Setup

Install dependencies and configure API credentials.

In [ ]:
# For Google Colab - clone repo and install
# !git clone https://github.com/cgftinc/synthetic-data-prep-lib.git
# %cd synthetic-data-prep-lib
# %pip install -e .[dev] turbopuffer

In [ ]:
# Local development setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

repo_root = Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
from synthetic_data_prep.utils import ensure_safe_python_version

ensure_safe_python_version()

In [ ]:
# Configuration

# Turbopuffer credentials — get your API key at https://turbopuffer.com/
TURBOPUFFER_API_KEY = "[INSERT API KEY]"
TURBOPUFFER_REGION = "aws-us-east-1"
NAMESPACE = "my-docs"

# CGFT API key — create one at https://app.cgft.io/account/api-keys
API_KEY = "[INSERT API KEY]"
BASE_URL = "https://app.cgft.io"

# Dataset configuration

# this should point to a local directory containing the documents you want to use for dataset generation.
# go to docs.cgft.io/rag/synthetic_datagen for sample documents you can use.
DOCS_PATH = "./samples/posthog/docs/"

# QA generation configuration
CORPUS_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]
NUM_SINGLE_HOP = 15
NUM_MULTI_HOP = 5

# Optional: provide Azure OpenAI credentials to store vector embeddings alongside BM25.
# Set EMBED_API_KEY to None to use BM25 full-text search only (no vectors stored).
EMBED_API_KEY = None      # e.g. "4T3rpNzE..."
EMBED_ENDPOINT = None     # e.g. "https://my-resource.cognitiveservices.azure.com/"
EMBED_DEPLOYMENT = None   # e.g. "text-embedding-3-small"
EMBED_API_VERSION = "2024-12-01-preview"

## Step 1: Point to Dataset

Chunk your documents and upload to a Turbopuffer namespace in one line.

> If you already have a populated namespace you want to use, skip this step.

In [ ]:
from synthetic_data_prep.corpus.turbopuffer import TpufChunkSource

source = TpufChunkSource(api_key=TURBOPUFFER_API_KEY, namespace=NAMESPACE, region=TURBOPUFFER_REGION)

# Optional: build an embedding function to store vectors alongside BM25.
# Remove this block (or leave EMBED_API_KEY=None above) for BM25-only.
embed_fn = None
if EMBED_API_KEY:
    from openai import AzureOpenAI
    _openai = AzureOpenAI(api_key=EMBED_API_KEY, api_version=EMBED_API_VERSION, azure_endpoint=EMBED_ENDPOINT)
    def embed_fn(texts):
        response = _openai.embeddings.create(model=EMBED_DEPLOYMENT, input=texts)
        return [item.embedding for item in sorted(response.data, key=lambda x: x.index)]

source.populate_from_folder(DOCS_PATH, embed_fn=embed_fn)

## Step 2: Create Synthetic QA

Generate synthetic question-answer pairs from your corpus. This will take a few minutes, so go get a coffee :)

In [ ]:
from synthetic_data_prep.qa_generation import generate_dataset

dataset = generate_dataset(
    source=source,
    api_key=API_KEY,
    corpus_description=CORPUS_DESCRIPTION,
    example_queries=EXAMPLE_QUERIES,
    num_single_hop=NUM_SINGLE_HOP,
    num_multi_hop=NUM_MULTI_HOP,
)

## Step 3: Upload Dataset & Create Environment

Upload the dataset and bundle the Turbopuffer search environment for training.

In [ ]:
import synthetic_data_prep
from synthetic_data_prep.envs.tpuf_search_env import TpufSearchEnv
from synthetic_data_prep.trainer.pipeline import train

experiment_id = train(
    env_class=TpufSearchEnv,
    env_args={
        "turbopuffer_api_key": TURBOPUFFER_API_KEY,
        "namespace": NAMESPACE,
        "region": TURBOPUFFER_REGION,
    },
    dataset=dataset.to_list(),
    api_key=API_KEY,
    local_modules=[synthetic_data_prep],
    pip_dependencies=["turbopuffer"],
)

## Monitoring Training: What to Expect

### Early Training Noise

**Early training**: At the start of a training run, rewards will fluctuate and metrics may be noisy. This is completely normal - the model is still learning basic patterns and its outputs are unstable. Give it some time (usually a few dozen steps) and the signal will clean up and you'll start seeing clear trends.

**Exploration before exploitation**: Ideally, you want to see pass@k climbing first before average reward increases significantly. This means your model is exploring the solution space and learning to solve more prompts (breadth) before it optimizes for higher rewards on those prompts (depth). If average reward shoots up while pass@k stays low, you're likely exploiting a narrow set of easy prompts rather than building robust capabilities.

**Healthy training trajectory**: In a well-configured training run, expect pass@k to increase early as the model learns to solve more distinct prompts. Average reward should follow but lag behind. Eventually both should plateau as the model saturates your training distribution.

**Warning signs**:

- pass@k flat while average reward increases → model is overfitting to a narrow subset of prompts
- both metrics stagnate early → training distribution may be too hard, reward signal too sparse